In [ ]:
import zipfile
import os

In [ ]:
path_to_zip = '/content/drive/MyDrive/Colab Notebooks/5-iunie-dataset.zip'
if not os.path.isfile(path_to_zip):
  print('not a file')

In [ ]:
with zipfile.ZipFile(path_to_zip, 'r') as zip_ref:
    zip_ref.extractall("new_dataset")

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
import tensorflow as tf

In [ ]:
import cv2
from PIL import Image
import plotly.express as px

In [ ]:
path_to_dataset = 'new_dataset/5-iunie-dataset'

In [ ]:
os.listdir(path_to_dataset)

In [ ]:
for dir in os.listdir(path_to_dataset):
  dir_path = os.path.join(path_to_dataset, dir)
  if not os.path.isdir(dir_path):
    continue
  count = sum(1 for f in os.listdir(dir_path) if os.path.isfile(os.path.join(dir_path, f)))
  print(f'{dir_path} - {count}')

In [ ]:
def remove_ds_store(root_path):
    removed = 0
    for dirpath, dirnames, filenames in os.walk(root_path):
        for file in filenames:
            if file == '.DS_Store':
                file_path = os.path.join(dirpath, file)
                try:
                    os.remove(file_path)
                    removed += 1
                    print(f"Removed: {file_path}")
                except Exception as e:
                    print(f"Failed to remove {file_path}: {e}")
    if removed == 0:
        print("✅ No .DS_Store files found.")
    else:
        print(f"🧹 Removed {removed} .DS_Store files.")


remove_ds_store(path_to_dataset)

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

In [ ]:
def split_with_sklearn(source_dir, target_dir, train_size=0.8, val_size=0.1, test_size=0.1):
    assert train_size + val_size + test_size == 1.0, "The sum must be 1"

    for label in os.listdir(source_dir):
        class_dir = os.path.join(source_dir, label)
        if not os.path.isdir(class_dir):
            continue

        images = [f for f in os.listdir(class_dir) if os.path.isfile(os.path.join(class_dir, f))]

        train_imgs, temp_imgs = train_test_split(images, train_size=train_size, random_state=27)
        val_imgs, test_imgs = train_test_split(temp_imgs, test_size=test_size / (val_size + test_size), random_state=27)

        for split, split_imgs in zip(['train', 'val', 'test'], [train_imgs, val_imgs, test_imgs]):
            split_dir = os.path.join(target_dir, split, label)
            os.makedirs(split_dir, exist_ok=True)
            for img in split_imgs:
                src = os.path.join(class_dir, img)
                dst = os.path.join(split_dir, img)
                shutil.copy2(src, dst)

source = 'new_dataset/5-iunie-dataset'
destination = 'dataset_split'
split_with_sklearn(source, destination)

In [ ]:
import plotly.express as px

In [ ]:
def plot_class_distribution(set_path, set_name):
    class_names = [name for name in os.listdir(set_path)
                   if os.path.isdir(os.path.join(set_path, name))]

    class_dis = [len(os.listdir(os.path.join(set_path, name))) for name in class_names]

    fig = px.bar(
        x=class_names,
        y=class_dis,
        color=class_names,
        title=f"Class Distribution - {set_name.capitalize()} Set"
    )
    fig.update_layout({'title': {'x': 0.5}})
    fig.show()

base_path = 'dataset_split'
train_path = os.path.join(base_path, 'train')
val_path = os.path.join(base_path, 'val')
test_path = os.path.join(base_path, 'test')

plot_class_distribution(train_path, 'train')
plot_class_distribution(val_path, 'val')
plot_class_distribution(test_path, 'test')

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [ ]:
os.listdir(train_path)

In [ ]:
train = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(260, 260),
    batch_size=32,
    shuffle=True
)

val = tf.keras.utils.image_dataset_from_directory(
    val_path,
    image_size=(260, 260),
    batch_size=32
)

test = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(260, 260),
    batch_size=32
)

In [ ]:
from tensorflow.keras.applications import EfficientNetB2
from tensorflow.keras import layers, models

In [ ]:
input_shape = (260, 260, 3)
num_classes = 13

base_model = tf.keras.applications.EfficientNetB2(
    include_top=False,
    weights='imagenet',
    input_shape=input_shape
)
base_model.trainable = False

inputs = layers.Input(shape=input_shape)
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.RMSprop(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train,
    validation_data=val,
    epochs=40
)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.RMSprop(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(train, validation_data=val, epochs=15)

In [ ]:
test_loss, test_acc = model.evaluate(test)
print(f"Test Accuracy: {test_acc:.2%}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='val acc')
plt.title('Training vs Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
y_pred = []
y_true = []

for images, labels in test:
    preds = model.predict(images)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

y_pred = np.array(y_pred)
y_true = np.array(y_true)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
class_names = test.class_names

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

plt.figure(figsize=(8, 6))
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title("Confusion Matrix - Test Set")
plt.grid(False)
plt.show()

In [ ]:
class_name = train.class_names
class_name

In [ ]:
import json

with open("class_labels.json", "w") as f:
    json.dump(class_names, f)

In [ ]:
model.save("Modul-2-iunie.keras")

In [ ]:
img = cv2.imread("dandelion.jpeg")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (224, 224))

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(img)
plt.show(), img.shape

In [ ]:
img_array = np.expand_dims(img, axis=0)
img_array = tf.keras.applications.efficientnet.preprocess_input(img_array)

In [ ]:
pred = model.predict(img_array)
predicted_index = np.argmax(pred)
predicted_label = class_names[predicted_index]

In [ ]:
print(f"Predicted class: {predicted_label} ({predicted_index})")